In [44]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, random_split
import tifffile as tiff
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from skimage import io
import sys
# from umap import UMAP
import joblib


# Load Data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [45]:
# Get the directory of the script
script_dir = os.getcwd()

# Get the parent directory of the script
parent_dir = os.path.dirname(script_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

from core.autoencoders import AE, train_ae

In [46]:
ae = torch.load('/mnt/d/lding/CLS_GitHub/fa_patch_AE_clustering/results/pax_ch1_ps32_latdim_16_20251001_1737/ae_model_at_min_val_loss_ep5967.pt', map_location=device, weights_only=False)
ae.eval()

AE(
  (encoder): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): LeakyReLU(negative_slope=0.01)
    (9): Flatten(start_dim=1, end_dim=-1)
  )
  (encoder_fc): Sequential(
    (0): Linear(in_features=2048, out_features=1024, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=1024, out_features=16, bias=True)
  )
  (decoder_fc): Sequential(
    (0): Linear(in_features=16, out_features=10

In [47]:
recon_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/' \
'ctrl_ch0_major/patches_gridonly_pslocation00/pax_ae_vin_recon_00_patches32_65p_20250909_1518'

os.makedirs(recon_dir, exist_ok=True)

raw_grid_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/' \
'ctrl_ch0_major/patches_gridonly_pslocation00/pax_ae_vin_raw_grid_00_patches32_65p_20250909_1518'
os.makedirs(raw_grid_dir, exist_ok=True)

raw_patch_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/' \
'ctrl_ch0_major/patches_gridonly_pslocation00/pax_ae_vin_raw_patch_00_patches32_65p_20250909_1518'
os.makedirs(raw_patch_dir, exist_ok=True)

recon_patch_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/' \
'ctrl_ch0_major/patches_gridonly_pslocation00/pax_ae_vin_recon_patch_00_patches32_65p_20250909_1518'
os.makedirs(recon_patch_dir, exist_ok=True)




In [48]:
def normalized_mse(x_hat, x):
    mse_loss = nn.MSELoss(reduction='mean')
    mse = mse_loss(x_hat, x)
    norm = torch.mean(x ** 2)  # average signal power
    return mse / norm
  
loss_fn_mseabs = nn.MSELoss(reduction='mean')
    
loss_fn_msenorm = normalized_mse

In [ ]:
csv_folder = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/' \
'ctrl_ch0_major/patches_gridonly_pslocation00/plot_patches32_65p_20250909_1518'

losses_mes_array=[]
losses_mse_norm_array=[]
group_mean

csv_filename = 'data_prep_record_49_t49.csv'

pad_size = 64
import pandas as pd

all_image_csv = pd.read_csv(os.path.join(csv_folder,csv_filename))

unique_vals = all_image_csv["filename"].unique()

for filename in unique_vals:
    this_file_csv = all_image_csv[all_image_csv["filename"]==filename]
    # print(csv_filename)
    raw_whole_image = np.zeros([1024,1024])
    recon_whole_image = np.zeros([1024,1024])    

    for index, row in this_file_csv.iterrows():        
        x_corner1 = int(row["x_corner1"])
        x_corner3 = int(row["x_corner3"])
        y_corner1 = int(row["y_corner1"])
        y_corner3 = int(row["y_corner3"])
        patch_name = row['crop_img_filename']    
        
        raw_patch = tiff.imread(os.path.join(row["movie_partitioned_data_dir"].replace('patches_localmax_rotation_pslocation00','patches_gridonly_pslocation00'),row["crop_img_filename"] ))
        normed_raw_patch = raw_patch.copy() * 240
        normed_raw_patch[normed_raw_patch > 254] = 254
        normed_raw_patch = normed_raw_patch/255
        
        tensor_patch = torch.from_numpy(normed_raw_patch)
        tensor_patch = tensor_patch.unsqueeze(0).unsqueeze(0)
        tensor_patch = tensor_patch.to(device)
        ae = ae.to(device)
        with torch.no_grad():
            recon_image, latent = ae(tensor_patch)


        recon_patch_filename = "recon_patch_"+patch_name+'.tif'
    
        tiff.imwrite(
            os.path.join(recon_patch_dir,recon_patch_filename),
            recon_image.squeeze().cpu().detach().numpy().astype(np.float32),
            imagej=True,              # Write ImageJ metadata block
            metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
        )    
        
        raw_patch_img_filename = "raw_patch_"+patch_name+'.tif'
        
        tiff.imwrite(
            os.path.join(raw_patch_dir,raw_patch_img_filename),
            normed_raw_patch.astype(np.float32),
            imagej=True,              # Write ImageJ metadata block
            metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
        )

        raw_whole_image[y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] = normed_raw_patch
        recon_whole_image[y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] = recon_image.squeeze().cpu().detach()

        loss_mse_value = loss_fn_mseabs(
            torch.from_numpy(normed_raw_patch.astype(np.float32)),
            torch.from_numpy(recon_image.squeeze().cpu().detach().numpy().astype(np.float32))
        ).item()

        loss_msenorm_value = loss_fn_msenorm(
            torch.from_numpy(normed_raw_patch.astype(np.float32)),
            torch.from_numpy(recon_image.squeeze().cpu().detach().numpy().astype(np.float32))
        ).item()


        losses_mes_array.append(loss_mse_value)
        losses_mse_norm_array.append(loss_msenorm_value)  

    # fig, ax = plt.subplots(1,2, figsize=(8,4))
    # ax[0].imshow(raw_whole_image,cmap=plt.cm.gray)
    # ax[1].imshow(recon_whole_image,cmap=plt.cm.gray)

    recon_img_filename = "recon_"+filename+'.tif'
    
    tiff.imwrite(
        os.path.join(recon_dir,recon_img_filename),
        recon_whole_image.astype(np.float32),
        imagej=True,              # Write ImageJ metadata block
        metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
    )    
    
    raw_grid_img_filename = "raw_grid_"+filename+'.tif'
    
    tiff.imwrite(
        os.path.join(raw_grid_dir,raw_grid_img_filename),
        raw_whole_image.astype(np.float32),
        imagej=True,              # Write ImageJ metadata block
        metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
    )

    # break

In [50]:
joblib.dump(
    {
        "losses_mes_array": losses_mes_array,
        "losses_mse_norm_array": losses_mse_norm_array
    },
    "../results/pax_ae_to_zyx.pkl"
)

['../results/pax_ae_to_zyx.pkl']

In [51]:
np.mean(losses_mes_array)

0.0033938006669272255

In [52]:
np.mean(losses_mse_norm_array)

2.788487716968859